# 02b · Experimentos de SfM: efecto de cada mejora

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StevenFAP1815/project-reconstruction-3d/blob/main/notebooks/02b_sfm_experiments.ipynb)

En la línea base (`02_sfm_colmap.ipynb`) la caja registró 43 de 68 fotos
(63 %) en 2 modelos, con un error de 1,11 px. El diagnóstico apuntó a fotos
subexpuestas (pocos puntos SIFT) y a un lado de la caja sin registrar.

Este notebook mide el efecto de cada mejora, **añadiéndolas de una en una**
sobre las mismas fotos (criterio 4: "muestra el efecto de cada paso"):

| Configuración | Cambio respecto a la anterior |
|---|---|
| `A_base` | Igual que la línea base. |
| `B_clahe` | Fotos con contraste mejorado (CLAHE), con la misma focal inicial del EXIF. |
| `C_sensible` | + SIFT más sensible, emparejamiento guiado y registro más permisivo. |
| `D_calibrada` | + intrínsecos fijos de la calibración con tablero. |

**Entorno:** GPU. Tiempo aproximado: 4-6 min por configuración.

## 1. Entorno

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/StevenFAP1815/project-reconstruction-3d.git"
REPO_DIR = Path("/content/project-reconstruction-3d")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

In [ ]:
%pip install -q -r /content/project-reconstruction-3d/requirements-colab.txt

## 2. Parámetros y datos

In [ ]:
OBJECT = "caja-carton-leche"
SEED = 42
BLUR_RATIO = 0.35
CLAHE_CLIP = 2.0

In [ ]:
import json
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycolmap

from recon3d.dataset import extract_object_zip
from recon3d.paths import get_paths
from recon3d.preprocessing import enhance_contrast, flag_blurry, laplacian_variance
from recon3d.reproducibility import set_seed

set_seed(SEED)
paths = get_paths()
paths.create()

LOCAL_IMAGES = Path("/content/data") / OBJECT
images = extract_object_zip(paths.data_zip(OBJECT), LOCAL_IMAGES)
sharpness = {p.name: laplacian_variance(p) for p in images}
blurry = flag_blurry(sharpness, ratio=BLUR_RATIO)
used = [p for p in images if p.name not in blurry]
print(f"{OBJECT}: {len(images)} fotos, {len(blurry)} borrosas, {len(used)} usadas")
print("pycolmap", pycolmap.__version__, "| CUDA:", pycolmap.has_cuda)

In [ ]:
# Dos carpetas de entrada: fotos originales y fotos con CLAHE (enlaces / copias).
WORK = Path("/content/work") / OBJECT / "experiments"
RAW_DIR = WORK / "images_raw"
CLAHE_DIR = WORK / "images_clahe"
for folder in (RAW_DIR, CLAHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
    for old in folder.iterdir():
        old.unlink()

for path in used:
    (RAW_DIR / path.name).symlink_to(path)
    enhance_contrast(path, CLAHE_DIR / path.name, clip_limit=CLAHE_CLIP)
print(f"{len(used)} fotos preparadas en {RAW_DIR.name} y {CLAHE_DIR.name}")

## 3. Efecto de CLAHE sobre una foto

Se compara la misma foto antes y después, y el número de puntos SIFT que
encuentra OpenCV en una muestra de fotos (a 1600 px, como referencia rápida).

In [ ]:
sample_name = used[len(used) // 2].name
raw = cv2.cvtColor(cv2.imread(str(RAW_DIR / sample_name)), cv2.COLOR_BGR2RGB)
enh = cv2.cvtColor(cv2.imread(str(CLAHE_DIR / sample_name)), cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(1, 2, figsize=(10, 7))
for ax, image, title in zip(axes, [raw, enh], ["Original", "CLAHE"]):
    ax.imshow(image)
    ax.set_title(f"{title}: {sample_name}")
    ax.axis("off")
plt.show()

sift = cv2.SIFT_create()
def count_sift(path, side=1600):
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    scale = side / max(gray.shape)
    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    return len(sift.detect(gray, None))

rng = np.random.default_rng(SEED)
sample = rng.choice([p.name for p in used], size=min(10, len(used)), replace=False)
sift_df = pd.DataFrame({
    "image": sample,
    "sift_original": [count_sift(RAW_DIR / n) for n in sample],
    "sift_clahe": [count_sift(CLAHE_DIR / n) for n in sample],
})
sift_df["aumento_%"] = 100 * (sift_df["sift_clahe"] / sift_df["sift_original"] - 1)
print(f"Puntos SIFT medios: {sift_df['sift_original'].mean():.0f} -> {sift_df['sift_clahe'].mean():.0f}")
sift_df.round(1)

## 4. Intrínsecos de la calibración

Se leen de `results/calibration/calibration.json` en Drive. Con la calibración
v2 (k3 = 0) el modelo de COLMAP es `OPENCV`.

In [ ]:
from recon3d.calibration import colmap_camera_from_calibration

calibration_file = paths.calibration_results / "calibration.json"
calibration_info = json.loads(calibration_file.read_text())
CALIB_MODEL, CALIB_PARAMS = colmap_camera_from_calibration(calibration_file)
print("Calibración:", "v2 (k3 fijo)" if calibration_info.get("fix_k3") else "v1",
      f"| RMS = {calibration_info['rms_px']} px")
print("Modelo COLMAP:", CALIB_MODEL)
print("Parámetros:", np.round(CALIB_PARAMS, 4))

## 5. Configuraciones y ejecución

In [ ]:
SENSITIVE = dict(
    max_features=16384,
    extraction_options={"sift.peak_threshold": 0.004},   # por defecto 0.0067: más puntos en zonas de poco contraste
    matching_options={"guided_matching": True},           # reutiliza la geometría para encontrar más coincidencias
    mapping_options={
        "mapper.abs_pose_min_num_inliers": 15,             # por defecto 30: registra fotos con menos coincidencias
        "mapper.abs_pose_min_inlier_ratio": 0.15,          # por defecto 0.25
    },
)

# CLAHE reescribe los JPEG y se pierde el EXIF, del que COLMAP toma la focal
# inicial. Para que la comparación con A sea justa, B y C reciben esa misma
# focal inicial (deducida del EXIF: 24 mm equivalentes) y la siguen refinando.
from recon3d.calibration import focal_px_from_35mm

EXIF_INIT = dict(camera_model="SIMPLE_RADIAL",
                 camera_params=[focal_px_from_35mm(24, (3000, 4000)), 1500.0, 2000.0, 0.0])

CONFIGS = {
    "A_base": dict(image_dir=RAW_DIR),
    "B_clahe": dict(image_dir=CLAHE_DIR, **EXIF_INIT),
    "C_sensible": dict(image_dir=CLAHE_DIR, **EXIF_INIT, **SENSITIVE),
    "D_calibrada": dict(image_dir=CLAHE_DIR, **SENSITIVE, camera_model=CALIB_MODEL,
                        camera_params=CALIB_PARAMS, refine_intrinsics=False),
}

In [ ]:
from recon3d.sfm import run_sfm, sfm_stats

runs, rows = {}, []
for name, config in CONFIGS.items():
    config = dict(config)
    image_dir = config.pop("image_dir")
    report = {}
    print(f"=== {name} ===")
    try:
        rec, timings, n_models = run_sfm(image_dir, WORK / name, report=report, **config)
    except Exception as error:  # una configuración fallida no detiene las demás
        print("  falló:", error)
        rows.append({"config": name, "error": str(error)})
        continue
    runs[name] = (rec, report)
    stats = sfm_stats(rec, n_images=len(used))
    rows.append({"config": name, **stats, "n_models": n_models,
                 "model_sizes": report["model_sizes"], "total_s": round(timings["total_s"], 1),
                 "ignored_options": report["ignored_options"]})
    print(f"  {stats['n_registered']}/{len(used)} registradas ({stats['registered_pct']} %), "
          f"{n_models} modelos {report['model_sizes']}, error {stats['mean_reprojection_error_px']} px")
    if report["ignored_options"]:
        print("  opciones no reconocidas por esta versión de pycolmap:", report["ignored_options"])

results_df = pd.DataFrame(rows)
results_df

In [ ]:
ok = results_df.dropna(subset=["registered_pct"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.bar(ok["config"], ok["registered_pct"])
ax1.axhline(90, color="red", linestyle="--", label="referencia 90 %")
ax1.set_ylabel("% fotos registradas")
ax1.legend()
ax2.bar(ok["config"], ok["mean_reprojection_error_px"])
ax2.axhline(1, color="red", linestyle="--", label="referencia 1 px")
ax2.set_ylabel("Error de reproyección (px)")
ax2.legend()
plt.suptitle(f"Efecto de cada mejora en {OBJECT}")
plt.show()

## 6. Mejor configuración

Criterio: más fotos registradas; a igualdad, menor error de reproyección.

In [ ]:
best_name = (ok.sort_values(["n_registered", "mean_reprojection_error_px"], ascending=[False, True])
               .iloc[0]["config"])
best_rec, best_report = runs[best_name]
print("Mejor configuración:", best_name)
print(best_rec.summary())
print(f"Fotos sin registrar ({len(best_report['unregistered'])}):", best_report["unregistered"])

In [ ]:
from recon3d.geometry import trajectory_frame
from recon3d.sfm import camera_centers

names, centers = camera_centers(best_rec)
coords, _ = trajectory_frame(centers)
fig, (ax_top, ax_side) = plt.subplots(1, 2, figsize=(13, 5))
sc = ax_top.scatter(coords[:, 0], coords[:, 1], c=coords[:, 2], cmap="viridis", s=25)
ax_top.set_aspect("equal")
ax_top.set_title("Vista desde arriba (color = altura)")
fig.colorbar(sc, ax=ax_top)
ax_side.scatter(np.hypot(coords[:, 0], coords[:, 1]), coords[:, 2], s=25)
ax_side.set_xlabel("distancia al eje")
ax_side.set_ylabel("altura")
ax_side.set_title("Vista lateral")
plt.suptitle(f"Trayectoria con {best_name}: {len(names)} cámaras")
plt.show()

In [ ]:
missing = best_report["unregistered"][:6]
if missing:
    fig, axes = plt.subplots(1, len(missing), figsize=(3 * len(missing), 4))
    for ax, name in zip(np.atleast_1d(axes), missing):
        ax.imshow(cv2.cvtColor(cv2.imread(str(CLAHE_DIR / name)), cv2.COLOR_BGR2RGB))
        ax.set_title(name, fontsize=7)
        ax.axis("off")
    plt.suptitle("Fotos sin registrar (con CLAHE)")
    plt.show()
else:
    print("Todas las fotos quedaron registradas.")

## 7. Guardar resultados

- `results/<objeto>/colmap-v2/`: modelo de la mejor configuración y su `meta.json`.
- `results/<objeto>/colmap-v2/experiments.csv`: tabla de todas las configuraciones.

In [ ]:
from recon3d.results import write_meta
from recon3d.sfm import save_sparse_results

run_dir = paths.run_dir(OBJECT, "colmap-v2")
save_sparse_results(best_rec, run_dir)
results_df.to_csv(run_dir / "experiments.csv", index=False)
best_row = ok[ok["config"] == best_name].iloc[0]
meta_path = write_meta(
    run_dir,
    object_name=OBJECT,
    method="colmap-v2",
    n_images=len(images),
    n_registered=best_rec.num_reg_images(),
    runtime_s=float(best_row["total_s"]),
    seed=SEED,
    extra={
        "best_config": best_name,
        "configs": {k: {kk: (str(vv) if isinstance(vv, Path) else vv) for kk, vv in v.items()}
                    for k, v in CONFIGS.items()},
        "blurry_images": blurry,
        "n_images_used": len(used),
        "clahe_clip": CLAHE_CLIP,
        "calibration": {"model": CALIB_MODEL, "params": CALIB_PARAMS,
                        "fix_k3": calibration_info.get("fix_k3", False)},
        "model_sizes": best_report["model_sizes"],
        "unregistered": best_report["unregistered"],
        "sift_original_mean": float(sift_df["sift_original"].mean()),
        "sift_clahe_mean": float(sift_df["sift_clahe"].mean()),
    },
)
print(run_dir)
print(meta_path)